# 04E - Entrenamiento y exportacion de los modelos oficiales (top-3)

Entrena y exporta **dos modelos** del top-3, con sus **hiperparametros exactos** de la busqueda V2,
sobre el **dataset final** (todas las variables predictoras = 64 features):

| Modelo | Overfitting | Hiperparametros |
|---|---|---|
| **XGBoost** | moderado (gap 0.055) | max_depth=8, n_estimators=600, lr=0.12, min_child_weight=5, reg_alpha=0.1, reg_lambda=5.0, subsample=0.95, colsample_bytree=0.95 |
| **Regresion Logistica** | bajo (gap ~0.0001) | C=0.3219596952967591, penalty=l2, solver=lbfgs |

Cada uno se exporta como un *bundle* que el backend carga; el usuario elige el modelo en la interfaz.

**Salidas** en `modelo/V2/06_ARTEFACTOS/`:
- `xgboost_susceptibility_v2.joblib` (modelo por defecto)
- `logreg_susceptibility_v2.joblib`

Ejecutar con el kernel `Python (armd-ai)`.

## 1. Librerias

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
import shap

print('Librerias cargadas. shap', shap.__version__)

Librerias cargadas. shap 0.51.0


## 2. Rutas, constantes e hiperparametros del top-3

In [2]:
RAIZ = Path.cwd()
while RAIZ.name != 'Proyecto' and RAIZ.parent != RAIZ:
    RAIZ = RAIZ.parent

DATASET = RAIZ / 'modelo' / 'V2' / 'DATOS_PROCESADOS' / '09_dataset_v2_multibacteria_balanceado_organismo_clase.csv'
ARTEFACTOS = RAIZ / 'modelo' / 'V2' / '06_ARTEFACTOS'
ARTEFACTOS.mkdir(parents=True, exist_ok=True)

TARGET = 'susceptibility'
CATEGORICAL = ['ordering_mode', 'culture_description', 'organism', 'antibiotic', 'age', 'gender']
IDENTIFICADORES = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']
CLASES = ['Susceptible', 'Resistant']  # indice 0 -> Susceptible, 1 -> Resistant

# Hiperparametros exactos del top-3 (busqueda V2)
XGB_PARAMS = dict(
    n_estimators=600, max_depth=8, learning_rate=0.12, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=5.0, subsample=0.95, colsample_bytree=0.95,
    objective='binary:logistic', eval_metric='logloss', tree_method='hist',
    n_jobs=-1, random_state=42,
)
LOGREG_PARAMS = dict(
    C=0.3219596952967591, penalty='l2', solver='lbfgs', fit_intercept=True,
    tol=0.0001, max_iter=1000, n_jobs=-1,
)
print('Dataset:', DATASET)

Dataset: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\V2\DATOS_PROCESADOS\09_dataset_v2_multibacteria_balanceado_organismo_clase.csv


## 3. Seleccion de variables (todo el dataset menos IDs y target)

In [3]:
columnas = pd.read_csv(DATASET, nrows=0).columns.tolist()
excluir = set(IDENTIFICADORES) | {TARGET}
features = [c for c in columnas if c not in excluir]
numericas = [c for c in features if c not in CATEGORICAL]
print(f'Features: {len(features)} | numericas: {len(numericas)} | categoricas: {len(CATEGORICAL)}')

Features: 64 | numericas: 58 | categoricas: 6


## 4. Carga, preparacion y particion

In [4]:
df = pd.read_csv(DATASET, usecols=features + [TARGET])
df = df[df[TARGET].isin(CLASES)].copy()
for c in numericas:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for c in CATEGORICAL:
    df[c] = df[c].astype('string').fillna('Null')

X = df[features]
y = (df[TARGET] == 'Resistant').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

C:\Users\Johan\AppData\Local\Temp\ipykernel_20476\1703298435.py:1: DtypeWarning: Columns (0: hosp_ward_ICU) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET, usecols=features + [TARGET])


Train: (194338, 64) | Test: (48585, 64)


## 5. Funciones auxiliares (preprocesador, evaluacion y exportacion)

- **XGBoost (tree):** One-Hot + imputacion por mediana (los arboles no requieren escalado).
- **Regresion Logistica (linear):** One-Hot + imputacion + `StandardScaler` (un modelo lineal si requiere escalar).
  Ademas se guarda un `shap_background` para que el backend use `shap.LinearExplainer`.

In [5]:
def construir_preprocesador(model_type):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if model_type == 'linear':
        num_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), CATEGORICAL),
            ('num', Pipeline(num_steps), numericas),
        ],
        remainder='drop', verbose_feature_names_out=True,
    )

def evaluar(pipeline, nombre):
    pred = pipeline.predict(X_test)
    print(f'\n=== {nombre} ===')
    print('F1 macro     :', round(f1_score(y_test, pred, average='macro'), 4))
    print('Balanced acc :', round(balanced_accuracy_score(y_test, pred), 4))
    print(classification_report(y_test, pred, target_names=CLASES))

def exportar(pipeline, model_type, archivo):
    bundle = {
        'pipeline': pipeline, 'feature_columns': features,
        'categorical': CATEGORICAL, 'numeric': numericas,
        'classes': CLASES, 'model_type': model_type,
    }
    if model_type == 'linear':
        bg = pipeline[:-1].transform(X_train.sample(min(200, len(X_train)), random_state=42))
        if hasattr(bg, 'toarray'):
            bg = bg.toarray()
        bundle['shap_background'] = np.asarray(bg, dtype='float64')
    joblib.dump(bundle, ARTEFACTOS / archivo)
    (ARTEFACTOS / archivo.replace('.joblib', '_metadata.json')).write_text(
        json.dumps({'model_type': model_type, 'n_features': len(features),
                    'feature_columns': features, 'classes': CLASES, 'target': TARGET,
                    'dataset': DATASET.name}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Exportado:', ARTEFACTOS / archivo)

## 6. XGBoost (overfitting moderado) - modelo por defecto

In [6]:
xgb_pipeline = Pipeline([('pre', construir_preprocesador('tree')), ('clf', XGBClassifier(**XGB_PARAMS))])
xgb_pipeline.fit(X_train, y_train)
evaluar(xgb_pipeline, 'XGBoost (top-3, overfit moderado)')

# Verificacion SHAP (TreeExplainer, igual que el backend)
Xt = xgb_pipeline[:-1].transform(X_test.iloc[:50])
if hasattr(Xt, 'toarray'): Xt = Xt.toarray()
_ = shap.TreeExplainer(xgb_pipeline[-1]).shap_values(Xt)
print('SHAP TreeExplainer OK.')

exportar(xgb_pipeline, 'tree', 'xgboost_susceptibility_v2.joblib')


=== XGBoost (top-3, overfit moderado) ===
F1 macro     : 0.8459
Balanced acc : 0.8417
              precision    recall  f1-score   support

 Susceptible       0.87      0.90      0.89     30000
   Resistant       0.83      0.78      0.81     18585

    accuracy                           0.86     48585
   macro avg       0.85      0.84      0.85     48585
weighted avg       0.86      0.86      0.86     48585

SHAP TreeExplainer OK.
Exportado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\V2\06_ARTEFACTOS\xgboost_susceptibility_v2.joblib


## 7. Regresion Logistica (overfitting bajo)

In [7]:
logreg_pipeline = Pipeline([('pre', construir_preprocesador('linear')), ('clf', LogisticRegression(**LOGREG_PARAMS))])
logreg_pipeline.fit(X_train, y_train)
evaluar(logreg_pipeline, 'Regresion Logistica (top-3, overfit bajo)')

# Verificacion SHAP (LinearExplainer, igual que el backend)
bg = logreg_pipeline[:-1].transform(X_train.sample(200, random_state=42))
if hasattr(bg, 'toarray'): bg = bg.toarray()
Xt = logreg_pipeline[:-1].transform(X_test.iloc[:50])
if hasattr(Xt, 'toarray'): Xt = Xt.toarray()
_ = shap.LinearExplainer(logreg_pipeline[-1], bg).shap_values(Xt)
print('SHAP LinearExplainer OK.')

exportar(logreg_pipeline, 'linear', 'logreg_susceptibility_v2.joblib')

C:\Users\Johan\miniconda3\envs\armd-ai\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Johan\miniconda3\envs\armd-ai\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



=== Regresion Logistica (top-3, overfit bajo) ===
F1 macro     : 0.7571
Balanced acc : 0.7545
              precision    recall  f1-score   support

 Susceptible       0.81      0.83      0.82     30000
   Resistant       0.72      0.68      0.69     18585

    accuracy                           0.77     48585
   macro avg       0.76      0.75      0.76     48585
weighted avg       0.77      0.77      0.77     48585

SHAP LinearExplainer OK.
Exportado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\V2\06_ARTEFACTOS\logreg_susceptibility_v2.joblib


## 8. Cierre

Reinicia `uvicorn`: el backend cargara ambos artefactos. En la interfaz podras elegir entre
**XGBoost** (por defecto) y **Regresion Logistica**, y cada prediccion traera su SHAP real.